In [14]:
import pandas as pd 
from pathlib import Path

In [15]:
DATA_DIR = Path(r"D:\SQL Project\retail-sales-rfm-analysis\data")

df = pd.read_csv(DATA_DIR/"raw"/"online_retail_II_Combined.csv",
                 parse_dates=["InvoiceDate"])

# Removing the space 
df = df.rename(columns={"Customer ID":"CustomerID"})

df.shape

(1067371, 9)

In [16]:
# printing only the rows from this date interval 
overlap = df[(df["InvoiceDate"]>="2010-12-01")&(df["InvoiceDate"]<"2010-12-10")]

# checking the number of rows coming from each sheet 
print(overlap.groupby("source_sheet").size())

print(overlap.groupby("source_sheet")["InvoiceDate"].nunique())


source_sheet
2009-2010    22523
2010-2011    22523
dtype: int64
source_sheet
2009-2010    921
2010-2011    921
Name: InvoiceDate, dtype: int64


Goal - Confirm karna ki sheet 1 aur sheet 2 ki overlap rows poori tarah identical hain.

In [17]:
# splitting the overlap into two dataframe based on the source sheet 
a = overlap[overlap["source_sheet"]== "2009-2010"]
b = overlap[overlap["source_sheet"]== "2010-2011"]

# columns to compare except source sheet 
cols = ["Invoice", "StockCode", "Description", "Quantity",
        "InvoiceDate", "Price", "CustomerID", "Country"]

In [18]:
check = a[cols].merge(b[cols],how="outer", indicator=True)

print(check["_merge"].value_counts())

_merge
both          23221
left_only         0
right_only        0
Name: count, dtype: int64


Goal
- 1-9 Dec 2010 ki sirf ek copy rakhna, yaani sheet 1 (2009-2010) ki overlap rows hata dena.
- Warna December 2010 ka revenue double count hota. Ab hum confirm kar chuke hain ki dono copies identical hain, toh ek hataane se  kuch information nahi khoti.


In [19]:
# dropping the overlapping rows 
rows_before = len(df)

# True for the rows that lies in only sheet 2009-2010 and in the following date interval 
# Use a proper Boolean mask , so sum() could count the true values and give the number of rows to drop
drop_mask = ((df["source_sheet"]=="2009-2010") 
               &(df["InvoiceDate"]>="2010-12-01")
               &(df["InvoiceDate"]<"2010-12-10"))

print ("Rows to drop: ", drop_mask.sum())


df = df[~drop_mask].copy()

print("Rows before : ",rows_before, "Rows after : ", len(df))

Rows to drop:  22523
Rows before :  1067371 Rows after :  1044848


In [20]:
# quick check for that overlap window only come from one sheet
overlap_check = df[(df["InvoiceDate"]>="2010-12-01") & (df["InvoiceDate"]<"2010-12-10")]
print(overlap_check["source_sheet"].value_counts())

source_sheet
2010-2011    22523
Name: count, dtype: int64


Goal

- Ginna ki kitni rows mein CustomerID khaali hai
- Decision lene se pehle size pata hona chahiye. 1% rows ho toh alag baat hai, 20% ho toh alag.


In [21]:
#True where customerid is missing 
missing = df["CustomerID"].isna()

print("Missing rows : ",missing.sum() ," out of ", len(df))
print("Percent Missing: ", round(missing.mean() *100,2),"%")

# displaying the fews records with missing customerid 
# automatically display the first 5 rows of the dateframe with missing customerid 
df[missing].head()



Missing rows :  235287  out of  1044848
Percent Missing:  22.52 %


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country,source_sheet
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.00,NaN,United Kingdom,2009-2010
283,489463,71477,short,-240,2009-12-01 10:52:00,0.00,NaN,United Kingdom,2009-2010
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.00,NaN,United Kingdom,2009-2010
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.00,NaN,United Kingdom,2009-2010
577,489525,85226C,BLUE PULL BACK RACING CAR,1,2009-12-01 11:49:00,0.55,NaN,United Kingdom,2009-2010


Goal
- Ye samajhna ki missing-ID rows kitna revenue rakhti hain, aur inme kitni rows Price zero, negative Quantity, ya "C" invoice ki hain.
- Agar ye rows kaafi revenue rakhti hain, toh revenue analysis se hataana galat hoga. Agar inme zyada kachra hai (Price 0), toh alag decision banta hai.


In [22]:
# a temporary column to calculate the revenue for each row
df["Revenue"] = df["Quantity"]*df["Price"]

#compare rows and revenue : missing vs non missing 
print(df.groupby(missing).agg(rows = ("Invoice","size"),revenue = ("Revenue","sum")))     # rows/revenue are the names of the output column, false = not missing 

share = df.loc[missing,"Revenue"].sum()/df["Revenue"].sum()*100
print("Share of revenue from the missing customerid rows : ",round(share,2),"%")

              rows       revenue
CustomerID                      
False       809561  1.634367e+07
True        235287  2.566093e+06
Share of revenue from the missing customerid rows :  13.57 %


In [23]:
# looking only at the missing id rows 
m = df[missing]

print("Price is equals to 0: ",(m["Price"]==0).sum())
print("Quantity < 0: ",(m["Quantity"]<0).sum())
print("Invoice starts with C: ",(m["Invoice"].astype(str).str.startswith("C")).sum())


Price is equals to 0:  5954
Quantity < 0:  4111
Invoice starts with C:  719


Insight
- Missing CustomerID 22.5% rows hain par sirf 13.6% revenue. Inhe revenue analysis mein rakhenge aur RFM se hataenge

Goal
- Verify karna ki negative Quantity rows "C" wali hain ya bina "C" ki, aur bina "C" wali rows mein Price zero kitni hain
- Agar bina-"C" negative rows mein Price zero hai, toh ye sales ya returns nahi, bas stock entry hain. Unhe revenue se hataana sahi hoga.

In [24]:
# all rows with the negative quantity (whole data)
negative = df[df["Quantity"]<0]

# true where the invoice starts with c 
is_c = negative["Invoice"].astype(str).str.startswith("C")

print("Negative quantity rows : ", len(negative))
print("Rows with C :", is_c.sum())
print("Rows without C :", (~is_c).sum())

non_c = negative[~is_c]
print("Rows without C and price = 0: ",(non_c["Price"]==0).sum())
non_c.head(5)

Negative quantity rows :  22557
Rows with C : 19164
Rows without C : 3393
Rows without C and price = 0:  3393


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country,source_sheet,Revenue
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom,2009-2010,-0.0
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom,2009-2010,-0.0
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom,2009-2010,-0.0
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom,2009-2010,-0.0
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom,2009-2010,-0.0


Insight
- Negative Quantity ki 22557 rows mein se 19164 'C' wali (customer returns) hain aur 3393 bina 'C' ke, jinki Price 0 hai (stock adjustments).

Goal 
- data ko 3 hisson mein batenge 
1. data containing return invoice starts with C
2. data containing quantity > 0 and price > 0 and non C invoices 
3. the rest of the data 

In [25]:
# flag
# this will create a new col called is_cancel and will be true for the rows where the invoice starts with C 
df["Is_cancel"] = df["Invoice"].astype(str).str.startswith("C")

# group 1 (invoice starts with C)
returns = df[df["Is_cancel"]== True].copy()

# group 2 (Non C invoices and price and qunantity being above 0)

sales = df [(~df["Is_cancel"])&
            (df["Quantity"]>0)&
            (df["Price"]>0)].copy()

# group 3 all the others (stock entry and price < 0)
others = df[~df.index.isin(sales.index) & ~df.index.isin(returns.index)].copy()

print ("Returns :", len(returns))
print ("Sales :", len(sales))
print ("Others :", len(others))
print("Total : ",len(returns)+len(sales)+len(others), "| df rows: ",len(df))


Returns : 19165
Sales : 1019654
Others : 6029
Total :  1044848 | df rows:  1044848


Insight
- 1,044,848 rows ko 3 groups mein baanta: Sales 1,019,654, Returns 19,165, Other 6,029. Total match hua, koi row nahi khoi

Goal
- other ki 6,029 rows ko samajhna aur wo ek ajeeb "C" row dhoondhna.
- Returns 19,165 hai par "C" wali negative Quantity rows 19,164 thi. Matlab ek "C" invoice ki Quantity negative nahi hai.

In [33]:
# understanding the type of rows in the others dataframe 
print("Price == 0 : ",(others["Price"]==0).sum())
print("Price < 0 : ",(others["Price"]<0).sum())
print("Quantity < 0 : ",(others["Quantity"]<0).sum())
print("Quantity > 0 : ",(others["Quantity"]>0).sum())

#looking the rows with negative price 
print(others[others["Price"]<0])

Price == 0 :  6024
Price < 0 :  5
Quantity < 0 :  3393
Quantity > 0 :  2636
        Invoice StockCode      Description  Quantity         InvoiceDate  \
179403  A506401         B  Adjust bad debt         1 2010-04-29 13:36:00   
276274  A516228         B  Adjust bad debt         1 2010-07-19 11:24:00   
403472  A528059         B  Adjust bad debt         1 2010-10-20 12:04:00   
825444  A563186         B  Adjust bad debt         1 2011-08-12 14:51:00   
825445  A563187         B  Adjust bad debt         1 2011-08-12 14:52:00   

           Price  CustomerID         Country source_sheet   Revenue  Is_cancel  
179403 -53594.36         NaN  United Kingdom    2009-2010 -53594.36      False  
276274 -44031.79         NaN  United Kingdom    2009-2010 -44031.79      False  
403472 -38925.87         NaN  United Kingdom    2009-2010 -38925.87      False  
825444 -11062.06         NaN  United Kingdom    2010-2011 -11062.06      False  
825445 -11062.06         NaN  United Kingdom    2010-2011 -110

In [36]:
# finding the odd rows , a positive quantity C invoice 
odd = returns[returns["Quantity"]>=0]
odd

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country,source_sheet,Revenue,Is_cancel
76799,C496350,M,Manual,1,2010-02-01 08:24:00,373.57,NaN,United Kingdom,2009-2010,373.57,True


insight 
- Other group ki 6,029 rows mein 6,024 Price = 0 hain aur 5 'Adjust bad debt' entries hain (lagbhag -158K). Ye sales nahi hain, isliye revenue se bahar rakhi

Goal
- dekhna ki Price = 0 aur Quantity > 0 wali rows kya hain.
- Price = 0 ke saath Quantity positive ho toh ye free samples, damaged maal ki entry, ya data entry galti ho sakti hai. Description dekh ke decide karna hai ki inhe kachra maane ya nahi.




In [ ]:
# in the others checking for the price being zero and but quantiy being positive 
zero_pos = others[(others["Price"]==0)&
                  (others["Quantity"]>0)]

print("Rows with Price == 0 & Quantity >0: ",len(zero_pos))

# top 10 common description of these products 
print(zero_pos["Description"].value_counts(dropna=False).head(10))


Rows with Price <0 & Quantity >0:  2631
Description
NaN                              1642
check                              39
found                              28
OWL DOORSTOP                       14
adjustment                         14
POLYESTER FILLER PAD 45x45cm       12
POLYESTER FILLER PAD 40x40cm       10
?                                   9
Found                               9
FRENCH BLUE METAL DOOR SIGN 1       9
Name: count, dtype: int64
